In [1]:
from google.colab import drive
drive.mount('/content/drive')

import tensorflow as tf
print("TF:", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU'))


Mounted at /content/drive
TF: 2.20.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
import numpy as np
import pandas as pd
import os
import time
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras.layers import *
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.applications import DenseNet201
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint

from sklearn.metrics import (classification_report, confusion_matrix,
                              recall_score, fbeta_score)
print("All imports done")

All imports done


In [3]:
TRAIN_PATH = "/content/drive/MyDrive/Cataract/Data/Train"
TEST_PATH  = "/content/drive/MyDrive/Cataract/Data/Test"
MODEL_PATH = "/content/drive/MyDrive/Cataract/DenseNet201_FineTune.h5"

print("Train:", TRAIN_PATH)
print("Test: ", TEST_PATH)
print("Model:", MODEL_PATH)

Train: /content/drive/MyDrive/Cataract/Data/Train
Test:  /content/drive/MyDrive/Cataract/Data/Test
Model: /content/drive/MyDrive/Cataract/DenseNet201_FineTune.h5


In [4]:
print("Train folders:", os.listdir(TRAIN_PATH))
print("Test folders: ", os.listdir(TEST_PATH))

Train folders: ['Cataract', 'Normal', 'Not Eye']
Test folders:  ['Cataract', 'Normal', 'Not Eye']


In [5]:
datagen = ImageDataGenerator(
    rescale          = 1. / 255,
    validation_split = 0.2,
    horizontal_flip  = True,
    vertical_flip    = True
)
print(" Datagen ready")

 Datagen ready


In [6]:
train_it = datagen.flow_from_directory(
    TRAIN_PATH,
    target_size = (224, 224),
    color_mode  = 'rgb',
    class_mode  = 'categorical',
    batch_size  = 32,
    subset      = "training"
)

val_it = datagen.flow_from_directory(
    TRAIN_PATH,
    target_size = (224, 224),
    color_mode  = 'rgb',
    class_mode  = 'categorical',
    batch_size  = 32,
    subset      = "validation"
)

test_it = datagen.flow_from_directory(
    TEST_PATH,
    target_size = (224, 224),
    color_mode  = 'rgb',
    class_mode  = 'categorical',
    batch_size  = 32,
    shuffle     = False
)

print("Class indices:", train_it.class_indices)

Found 8896 images belonging to 3 classes.
Found 2221 images belonging to 3 classes.
Found 2552 images belonging to 3 classes.
Class indices: {'Cataract': 0, 'Normal': 1, 'Not Eye': 2}


In [7]:
base_model = DenseNet201(
    weights     = 'imagenet',
    input_shape = (224, 224, 3),
    include_top = False
)
print("DenseNet201 base loaded")

74836368/74836368 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
DenseNet201 base loaded


In [8]:
ocl1 = Conv2D(32, (3, 3), activation='relu')(base_model.output)
bn1  = BatchNormalization()(ocl1)
mp1  = MaxPooling2D(pool_size=(2, 2))(bn1)
do1  = Dropout(0.17)(mp1)

ocl2 = Conv2D(64, (2, 2), activation='relu')(do1)
bn2  = BatchNormalization()(ocl2)

al1  = GlobalAveragePooling2D()(bn2)

fc1  = Dense(64, activation='relu')(al1)
fc2  = Dense(32, activation='relu')(fc1)
fc3  = Dense(32, activation='relu')(fc2)

al2  = BatchNormalization()(fc3)
all2 = Dropout(0.3)(al2)

output = Dense(3, activation='softmax', name='preds')(all2)
print(" CNN head built")

 CNN head built


In [9]:
Cataract_Model = Model(inputs=base_model.input, outputs=output)
Cataract_Model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ zero_padding2d      │ (None, 230, 230,  │          0 │ input_layer[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, 112, 112,  │      9,408 │ zero_padding2d[0… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_bn            │ (None, 112, 112,  │        256 │ conv1_conv[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_relu          │ (None, 112, 112,  │          0 │ conv1_bn[0][0]    │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ zero_padding2d_1    │ (None, 114, 114,  │          0 │ conv1_relu[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1               │ (None, 56, 56,    │          0 │ zero_padding2d_1… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_bn   │ (None, 56, 56,    │        256 │ pool1[0][0]       │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_relu │ (None, 56, 56,    │          0 │ conv2_block1_0_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, 56, 56,    │      8,192 │ conv2_block1_0_r… │
│ (Conv2D)            │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, 56, 56,    │        512 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, 56, 56,    │          0 │ conv2_block1_1_b… │
│ (Activation)        │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_conv │ (None, 56, 56,    │     36,864 │ conv2_block1_1_r… │
│ (Conv2D)            │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_concat │ (None, 56, 56,    │          0 │ pool1[0][0],      │
│ (Concatenate)       │ 96)               │            │ conv2_block1_2_c… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block2_0_bn   │ (None, 56, 56,    │        384 │ conv2_block1_con… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block2_0_relu │ (None, 56, 56,    │          0 │ conv2_block2_0_b… │
│ (Activation)        │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block2_1_conv │ (None, 56, 56,    │     12,288 │ conv2_block2_0_r

 Total params: 18,891,139 (72.06 MB)

 Trainable params: 18,661,827 (71.19 MB)

 Non-trainable params: 229,312 (895.75 KB)

In [13]:
Cataract_Model.compile(
    optimizer = Adam(learning_rate=0.0001),
    loss      = 'categorical_crossentropy',
    metrics   = ['accuracy']
)
print(" Model compiled")

 Model compiled


In [ ]:
for ix in range(708):
    Cataract_Model.layers[ix].trainable = False

print("✅ First 708 layers frozen")

✅ First 708 layers frozen


In [ ]:
for i, layer in enumerate(Cataract_Model.layers):
    print(i, layer.name, layer.trainable)

0 input_layer False
1 zero_padding2d False
2 conv1_conv False
3 conv1_bn False
4 conv1_relu False
5 zero_padding2d_1 False
6 pool1 False
7 conv2_block1_0_bn False
8 conv2_block1_0_relu False
9 conv2_block1_1_conv False
10 conv2_block1_1_bn False
11 conv2_block1_1_relu False
12 conv2_block1_2_conv False
13 conv2_block1_concat False
14 conv2_block2_0_bn False
15 conv2_block2_0_relu False
16 conv2_block2_1_conv False
17 conv2_block2_1_bn False
18 conv2_block2_1_relu False
19 conv2_block2_2_conv False
20 conv2_block2_concat False
21 conv2_block3_0_bn False
22 conv2_block3_0_relu False
23 conv2_block3_1_conv False
24 conv2_block3_1_bn False
25 conv2_block3_1_relu False
26 conv2_block3_2_conv False
27 conv2_block3_concat False
28 conv2_block4_0_bn False
29 conv2_block4_0_relu False
30 conv2_block4_1_conv False
31 conv2_block4_1_bn False
32 conv2_block4_1_relu False
33 conv2_block4_2_conv False
34 conv2_block4_concat False
35 conv2_block5_0_bn False
36 conv2_block5_0_relu False
37 conv2_block

In [ ]:
mc = ModelCheckpoint(
    MODEL_PATH,
    monitor        = 'val_accuracy',
    mode           = 'max',
    verbose        = 1,
    save_best_only = True
)

history = Cataract_Model.fit(
    train_it,
    epochs          = 20,
    validation_data = val_it,
    callbacks       = [mc]
)
print("✅ Training done")

Epoch 1/20
278/278 ━━━━━━━━━━━━━━━━━━━━ 0s 9s/step - accuracy: 0.3999 - loss: 1.3981
Epoch 1: val_accuracy improved from None to 0.63800, saving model to /content/drive/MyDrive/Cataract/DenseNet201_FineTune.h5



Epoch 1: finished saving model to /content/drive/MyDrive/Cataract/DenseNet201_FineTune.h5
278/278 ━━━━━━━━━━━━━━━━━━━━ 3074s 11s/step - accuracy: 0.4482 - loss: 1.2550 - val_accuracy: 0.6380 - val_loss: 0.8322
Epoch 2/20
278/278 ━━━━━━━━━━━━━━━━━━━━ 0s 192ms/step - accuracy: 0.5510 - loss: 0.9883
Epoch 2: val_accuracy improved from 0.63800 to 0.70599, saving model to /content/drive/MyDrive/Cataract/DenseNet201_FineTune.h5



Epoch 2: finished saving model to /content/drive/MyDrive/Cataract/DenseNet201_FineTune.h5
278/278 ━━━━━━━━━━━━━━━━━━━━ 78s 281ms/step - accuracy: 0.5786 - loss: 0.9511 - val_accuracy: 0.7060 - val_loss: 0.6943
Epoch 3/20
278/278 ━━━━━━━━━━━━━━━━━━━━ 0s 198ms/step - accuracy: 0.6432 - loss: 0.8205
Epoch 3: val_accuracy improved from 0.70599 to 0.76182, saving model to /content/drive/MyDrive/Cataract/DenseNet201_FineTune.h5



Epoch 3: finished saving model to /content/drive/MyDrive/Cataract/DenseNet201_FineTune.h5
278/278 ━━━━━━━━━━━━━━━━━━━━ 73s 264ms/step - accuracy: 0.6496 - loss: 0.8040 - val_accuracy: 0.7618 - val_loss: 0.5850
Epoch 4/20
278/278 ━━━━━━━━━━━━━━━━━━━━ 0s 194ms/step - accuracy: 0.6930 - loss: 0.7074
Epoch 4: val_accuracy improved from 0.76182 to 0.77353, saving model to /content/drive/MyDrive/Cataract/DenseNet201_FineTune.h5



Epoch 4: finished saving model to /content/drive/MyDrive/Cataract/DenseNet201_FineTune.h5
278/278 ━━━━━━━━━━━━━━━━━━━━ 80s 257ms/step - accuracy: 0.6972 - loss: 0.6938 - val_accuracy: 0.7735 - val_loss: 0.5294
Epoch 5/20
278/278 ━━━━━━━━━━━━━━━━━━━━ 0s 194ms/step - accuracy: 0.7166 - loss: 0.6617
Epoch 5: val_accuracy improved from 0.77353 to 0.79604, saving model to /content/drive/MyDrive/Cataract/DenseNet201_FineTune.h5


...


Epoch 15: finished saving model to /content/drive/MyDrive/Cataract/DenseNet201_FineTune.h5
278/278 ━━━━━━━━━━━━━━━━━━━━ 71s 253ms/step - accuracy: 0.8212 - loss: 0.4423 - val_accuracy: 0.8690 - val_loss: 0.3286
Epoch 16/20
278/278 ━━━━━━━━━━━━━━━━━━━━ 0s 191ms/step - accuracy: 0.8152 - loss: 0.4479
Epoch 16: val_accuracy improved from 0.86898 to 0.86988, saving model to /content/drive/MyDrive/Cataract/DenseNet201_FineTune.h5



Epoch 16: finished saving model to /content/drive/MyDrive/Cataract/DenseNet201_FineTune.h5
278/278 ━━━━━━━━━━━━━━━━━━━━ 71s 255ms/step - accuracy: 0.8212 - loss: 0.4403 - val_accuracy: 0.8699 - val_loss: 0.3246
Epoch 17/20
278/278 ━━━━━━━━━━━━━━━━━━━━ 0s 199ms/step - accuracy: 0.8192 - loss: 0.4492
Epoch 17: val_accuracy improved from 0.86988 to 0.87663, saving model to /content/drive/MyDrive/Cataract/DenseNet201_FineTune.h5



Epoch 17: finished saving model to /content/drive/MyDrive/Cataract/DenseNet201_FineTune.h5
278/278 ━━━━━━━━━━━━━━━━━━━━ 80s 288ms/step - accuracy: 0.8272 - loss: 0.4353 - val_accuracy: 0.8766 - val_loss: 0.2997
Epoch 18/20
278/278 ━━━━━━━━━━━━━━━━━━━━ 0s 193ms/step - accuracy: 0.8217 - loss: 0.4427
Epoch 18: val_accuracy improved from 0.87663 to 0.87888, saving model to /content/drive/MyDrive/Cataract/DenseNet201_FineTune.h5



Epoch 18: finished saving model to /content/drive/MyDrive/Cataract/DenseNet201_FineTune.h5
278/278 ━━━━━━━━━━━━━━━━━━━━ 71s 255ms/step - accuracy: 0.8300 - loss: 0.4272 - val_accuracy: 0.8789 - val_loss: 0.3029
Epoch 19/20
278/278 ━━━━━━━━━━━━━━━━━━━━ 0s 191ms/step - accuracy: 0.8363 - loss: 0.3969
Epoch 19: val_accuracy did not improve from 0.87888
278/278 ━━━━━━━━━━━━━━━━━━━━ 67s 239ms/step - accuracy: 0.8326 - loss: 0.4095 - val_accuracy: 0.8739 - val_loss: 0.3168
Epoch 20/20
278/278 ━━━━━━━━━━━━━━━━━━━━ 0s 192ms/step - accuracy: 0.8356 - loss: 0.4071
Epoch 20: val_accuracy improved from 0.87888 to 0.88834, saving model to /content/drive/MyDrive/Cataract/DenseNet201_FineTune.h5



Epoch 20: finished saving model to /content/drive/MyDrive/Cataract/DenseNet201_FineTune.h5
278/278 ━━━━━━━━━━━━━━━━━━━━ 71s 255ms/step - accuracy: 0.8363 - loss: 0.3987 - val_accuracy: 0.8883 - val_loss: 0.3036
✅ Training done


In [18]:
import os
from google.colab import drive

# Remount drive
drive.mount('/content/drive', force_remount=True)

# Check file
path = "/content/drive/MyDrive/Cataract/DenseNet201_FineTune.h5"
print("File exists:", os.path.exists(path))

# Show all files in Cataract folder
print("\nAll files in Cataract folder:")
for f in os.listdir("/content/drive/MyDrive/Cataract/"):
    print(" ", f)

Mounted at /content/drive
File exists: False

All files in Cataract folder:
  Cataract Detection.docx
  Data.zip
  Data
  MobileNetV2_FineTune.h5
  Xception_FineTune.h5
  ResNet152V2_FineTune.h5
  DenseNet201_FineTune1.keras
  EYE
  InceptionResNetV2_FineTune.h5
  DenseNet201_FineTune1.h5
  latency_InceptionResNetV2.npy


In [20]:
best_model = tf.keras.models.load_model(MODEL_PATH)
print(" Best model loaded from:", MODEL_PATH)

 Best model loaded from: /content/drive/MyDrive/Cataract/DenseNet201_FineTune.h5


In [ ]:
test_it.reset()
test_loss, test_accuracy = best_model.evaluate(test_it)
print("Test Loss:    ", round(test_loss, 4))
print("Test Accuracy:", round(test_accuracy, 4))

80/80 ━━━━━━━━━━━━━━━━━━━━ 1027s 13s/step - accuracy: 0.8879 - loss: 0.2843
Test Loss:     0.2843
Test Accuracy: 0.8879


In [ ]:
test_it.reset()
y_pred_probs = best_model.predict(test_it)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = test_it.classes

recall = recall_score(y_true, y_pred, average='weighted')
f2     = fbeta_score(y_true, y_pred, beta=2, average='weighted')

print("Recall:  ", round(recall, 4))
print("F2 Score:", round(f2, 4))

print("\nClassification Report:")
print(classification_report(y_true, y_pred,
      target_names=list(test_it.class_indices.keys())))

print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred))

80/80 ━━━━━━━━━━━━━━━━━━━━ 63s 489ms/step
Recall:   0.8934
F2 Score: 0.8934

Classification Report:
              precision    recall  f1-score   support

    Cataract       0.85      0.84      0.85       800
      Normal       0.85      0.86      0.85       800
     Not Eye       0.97      0.96      0.97       952

    accuracy                           0.89      2552
   macro avg       0.89      0.89      0.89      2552
weighted avg       0.89      0.89      0.89      2552

Confusion Matrix:
[[676 108  16]
 [ 98 687  15]
 [ 18  17 917]]


In [21]:
import time
import numpy as np
import os

test_it.reset()
single_image_input = next(iter(test_it))[0][:1]
print(f"Input shape: {single_image_input.shape}  ← batch_size=1 (single image)")

print("Running warm-up inferences...")
for _ in range(10):
    best_model.predict(single_image_input, verbose=0)

N = 100
times = []
for _ in range(N):
    start = time.perf_counter()
    best_model.predict(single_image_input, verbose=0)
    end   = time.perf_counter()
    times.append((end - start) * 1000)

# ── Save 100 values to Drive ──────────────────────────────
np.save("/content/drive/MyDrive/Cataract/latency_DenseNet201.npy", np.array(times))
print("✅ Saved latency_DenseNet201.npy")

latency_mean = np.mean(times)
latency_std  = np.std(times)
latency_p95  = np.percentile(times, 95)
model_size_mb = os.path.getsize("/content/drive/MyDrive/Cataract/DenseNet201_FineTune.h5") / (1024*1024)

print("\n" + "="*55)
print("  DenseNet201 — Single-Image Latency Report")
print("="*55)
print(f"  Average Latency : {latency_mean:.2f} ± {latency_std:.2f} ms")
print(f"  P95 Latency     : {latency_p95:.2f} ms")
print(f"  Model File Size : {model_size_mb:.2f} MB")
print(f"  Benchmark Runs  : {N}")
print(f"  Input Shape     : {single_image_input.shape}")
print(f"  Hardware        : Google Colab T4 GPU")
print("="*55)

Input shape: (1, 224, 224, 3)  ← batch_size=1 (single image)
Running warm-up inferences...
✅ Saved latency_DenseNet201.npy

  DenseNet201 — Single-Image Latency Report
  Average Latency : 103.87 ± 12.22 ms
  P95 Latency     : 113.57 ms
  Model File Size : 74.31 MB
  Benchmark Runs  : 100
  Input Shape     : (1, 224, 224, 3)
  Hardware        : Google Colab T4 GPU
